# 03 - Dataset Exploration and Quality Audit (SQuAD v2)

## Definition
EDA (Exploratory Data Analysis) validates whether a dataset is suitable for retrieval and generation tasks.

## Theory
RAG quality depends on corpus diversity, metadata quality, and clean train/eval split design.

## Motivation
Weak datasets create misleading retrieval metrics and unstable generation quality.

## Architecture
Data acquisition -> deduplication -> split policy -> quality checks -> artifact persistence.

## Real-world Examples
- Enterprise support assistant over product docs
- Compliance assistant over policy text
- Research assistant over paper abstracts

## Best Practices
- Measure retrieval and generation separately
- Track latency and grounding together
- Keep prompts strict about citations and uncertainty

## Common Mistakes
- Skipping retrieval diagnostics
- Using mismatched embedding models for index/query
- Reporting only one metric and claiming broad quality


In [1]:
from pathlib import Path
import json
import pandas as pd

from rag_system import (
    RAGConfig,
    ProjectRunner,
    ChunkingStrategy,
)

cfg = RAGConfig(project_root=Path(".."), profile="balanced")
runner = ProjectRunner(cfg)


In [2]:
bundle = runner.ingest_dataset()
summary = bundle['summary']
summary


{'num_documents': 20233,
 'num_queries': 11873,
 'num_unique_titles': 477,
 'doc_length_mean': 739.6197795680324,
 'doc_length_median': 679.0,
 'doc_length_p95': 1315.0,
 'question_length_mean': 59.441169038996044,
 'question_length_p95': 98.0,
 'answer_length_mean': 20.916160593792174,
 'answerable_ratio': 0.4992840899519919,
 'unanswerable_ratio': 0.5007159100480081,
 'query_split_counts': {'validation': 11873}}

In [3]:
from rag_system.data import documents_to_frame, queries_to_frame
from rag_system.visualization import plot_document_lengths
from pathlib import Path


doc_df = documents_to_frame(bundle['documents'])
query_df = queries_to_frame(bundle['queries'])

fig_dir = Path('../data/artifacts/figures')
fig_dir.mkdir(parents=True, exist_ok=True)
plot_document_lengths(doc_df, fig_dir / '03_doc_lengths.png')

{
    'documents_shape': doc_df.shape,
    'queries_shape': query_df.shape,
    'answerable_ratio': summary['answerable_ratio'],
    'query_split_counts': summary['query_split_counts'],
    'leakage_audit': bundle.get('leakage_audit', {}),
    'figure': str(fig_dir / '03_doc_lengths.png'),
}


{'documents_shape': (20233, 6),
 'queries_shape': (11873, 8),
 'answerable_ratio': 0.4992840899519919,
 'query_split_counts': {'validation': 11873},
 'leakage_audit': {'corpus_splits': ['train', 'validation'],
  'eval_splits': ['validation'],
  'num_corpus_context_keys': 20233,
  'num_eval_context_keys': 1204,
  'split_context_key_overlap': 1204,
  'overlap_expected_for_retrieval': True,
  'split_contamination_risk': False,
  'missing_gold_doc_references': 0,
  'leakage_pass': True},
 'figure': '../data/artifacts/figures/03_doc_lengths.png'}

## Findings and Interpretation
Use these outputs to justify dataset quality and confirm split-aware evaluation integrity.